# Kalshi Preliminary EDA

## Kalshi Market EDA
w/ open API

kalshi heirarchical structure follows series -> event -> markets

first exploring related different series

In [1]:
import requests

# Get series information for KXHIGHNY
url = "https://api.elections.kalshi.com/trade-api/v2/series"
response = requests.get(url)
markets_data = response.json()


In [2]:
series_ticket_map = {}
titles = []
tickers = []
for series in markets_data['series']:
    print(series['ticker'], series['title'])
    series_ticket_map[series['title']] = series['ticker']
    titles.append(series['title'])
    tickers.append(series['ticker'])

print(len(titles))
print(len(tickers))


KXPA3D PA 3 Democratic primary
KXHFHOUSING Bill taxing/banning hedge funds owning single-family homes
KXMLBOAK Oakland Athletics relocate
KXGOVFALLJAPAN Gov fall Japan
KXBAFTAEDIT BAFTA for Best Editing
SPOTIFYALBUM-TTPB-W TTPB week 1
KXTOP202025 Billboard Top 20 artists
KXWALTZCOUNT Waltz confirmation
KXNFLWINS-SF Pro football wins San Francisco
KXLEBRONPLAY LeBron Next Play
KXH5HUMAN Human to human transmission of H5
KXCSGOGAME Counter-Strike 2 Games
KXSENMAJORITY Senate Majority leader
AAPLWATCH Apple watch sales banned
KXCONGRESSTARIFFCOUNT How many Senators will vote for the Cantwell-Grassley Trade Review Act of 2025?
KXTENCOACH Tennessee Next Coach
KXPOSTMALONE da
KXMILLERCNN Stephen Miller appears on CNN
KXSPOTIFYSONGSFAMEISAGUN How many streams will Addison Raes Fame is a gun receive in its first week?
KXSNAPPRESTURKEY Snap presidential election Turkey
KXHARVPRES New Harvard President
KXPGAR2LEAD PGA Round 2 Leader
KXTXSENRPRIMARYMOV Margin of victory in the first round of the 

### Semantic Map of Series Titles

In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np
import umap
import plotly.express as px

texts = titles[:500]

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(texts, normalize_embeddings=True)

reducer = umap.UMAP(random_state=42)
points_2d = reducer.fit_transform(embeddings)

ids = list(range(1, len(texts) + 1))

fig = px.scatter(
    x=points_2d[:, 0],
    y=points_2d[:, 1],
    text=ids,
    hover_name=texts,
    title="Semantic map of input strings (numbered, hover for text)",
)
fig.update_traces(textposition="top center")
fig.update_layout(xaxis_title="Dim 1", yaxis_title="Dim 2")
fig.show()

print("Legend (point -> text):")
for i, t in zip(ids, texts):
    print(f"{i}: {t}")

/opt/homebrew/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Legend (point -> text):
1: PA 3 Democratic primary
2: Bill taxing/banning hedge funds owning single-family homes
3: Oakland Athletics relocate
4: Gov fall Japan
5: BAFTA for Best Editing
6: TTPB week 1
7: Billboard Top 20 artists
8: Waltz confirmation
9: Pro football wins San Francisco
10: LeBron Next Play
11: Human to human transmission of H5
12: Counter-Strike 2 Games
13: Senate Majority leader
14: Apple watch sales banned
15: How many Senators will vote for the Cantwell-Grassley Trade Review Act of 2025?
16: Tennessee Next Coach
17: da
18: Stephen Miller appears on CNN
19: How many streams will Addison Raes Fame is a gun receive in its first week?
20: Snap presidential election Turkey
21: New Harvard President
22: PGA Round 2 Leader
23: Margin of victory in the first round of the Texas Senate Republican primary?
24: x
25: UCL Goalscorer
26: NBA Draft Top 15 Pick
27: Fuga: Melodies of Steel 3 Metacritic score
28: After Nike, Jordan, and Adidas - what will be the top brand based on Tr

### Going through Markets by Series - Dealing with Pagination

In [4]:
import requests

def get_all_markets(series_ticker):

    all_markets = []
    cursor = None
    base_url = "https://api.elections.kalshi.com/trade-api/v2/markets"

    while True:
        # Build URL with cursor if we have one
        if series_ticker:
            url = f"{base_url}?series_ticker={series_ticker}"
        else:
            url = base_url

        if cursor:
            url += f"&cursor={cursor}"

        response = requests.get(url)
        try:
            data = response.json()
        except Exception as e:
            print("Failed to parse JSON:", e)
            print("Raw response text:", response.text)
            break

        # Debug: print keys found in the response
        print("Response keys:", list(data.keys()))

        # Check if 'markets' exists in response; otherwise, print and break
        if 'markets' not in data:
            print("KeyError: 'markets' not found in the response.")
            print("Full response received:", data)
            break

        for market in data['markets']:
            print(market['title'])

        # Add markets from this page
        all_markets.extend(data['markets'])

        # Check if there are more pages
        cursor = data.get('cursor')
        if not cursor:
            break

        print(f"Fetched {len(data['markets'])} markets, total: {len(all_markets)}")

    return all_markets

# Example usage
markets = get_all_markets('KXUCLGAME')
print(f"Total markets found: {len(markets)}")


Response keys: ['cursor', 'markets']
PSG vs Monaco Winner?
PSG vs Monaco Winner?
PSG vs Monaco Winner?
Real Madrid vs SL Benfica Winner?
Real Madrid vs SL Benfica Winner?
Real Madrid vs SL Benfica Winner?
Atalanta vs Dortmund Winner?
Atalanta vs Dortmund Winner?
Atalanta vs Dortmund Winner?
Juventus vs Galatasaray Winner?
Juventus vs Galatasaray Winner?
Juventus vs Galatasaray Winner?
Atletico vs Club Brugge Winner?
Atletico vs Club Brugge Winner?
Atletico vs Club Brugge Winner?
Newcastle vs Qarabag Winner?
Newcastle vs Qarabag Winner?
Newcastle vs Qarabag Winner?
Inter vs Bodoe/Glimt Winner?
Inter vs Bodoe/Glimt Winner?
Inter vs Bodoe/Glimt Winner?
Leverkusen vs Olympiacos Winner?
Leverkusen vs Olympiacos Winner?
Leverkusen vs Olympiacos Winner?
Bodoe/Glimt vs Inter Winner?
Bodoe/Glimt vs Inter Winner?
Bodoe/Glimt vs Inter Winner?
Olympiacos vs Leverkusen Winner?
Olympiacos vs Leverkusen Winner?
Olympiacos vs Leverkusen Winner?
Dortmund vs Atalanta Winner?
Dortmund vs Atalanta Winner?

## Working with API Keys

In [5]:
import requests
import datetime
import base64
from cryptography.hazmat.primitives import serialization, hashes
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives.asymmetric import padding
import os
import sys

# Ensure these are set in your environment or hardcode for testing purposes
API_KEY_ID = os.getenv("KALSHI_ACCESS_KEY")  # Env var uses underscore, NOT dash
PRIVATE_KEY_PATH = 'private-key.key'
BASE_URL = 'https://demo-api.kalshi.co'  # Change to 'https://api.kalshi.com' for production

def load_private_key(key_path):
    """Load the private key from file."""
    with open(key_path, "rb") as f:
        return serialization.load_pem_private_key(f.read(), password=None, backend=default_backend())

def create_signature(private_key, timestamp, method, path):
    """Create the request signature."""
    # Remove query params, Kalshi expects only path
    path_without_query = path.split('?')[0]
    message = f'{timestamp}{method}{path_without_query}'.encode("utf-8")
    signature = private_key.sign(
        message,
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.DIGEST_LENGTH
        ),
        hashes.SHA256()
    )
    return base64.b64encode(signature).decode("utf-8")

def get_bal(private_key, api_key_id, path, base_url=BASE_URL):
    """Make an authenticated GET request to Kalshi API."""
    timestamp = str(int(datetime.datetime.now(datetime.timezone.utc).timestamp() * 1000))
    signature = create_signature(private_key, timestamp, "GET", path)

    headers = {
        "Kalshi-Access-Key": api_key_id,
        "Kalshi-Access-Timestamp": timestamp,
        "Kalshi-Access-Signature": signature
    }
    response = requests.get(base_url + path, headers=headers)
    return response

# --- MAIN LOGIC ---

# Sanity checks
if API_KEY_ID is None:
    print("ERROR: Kalshi API key was not found in your environment variable 'KALSHI_ACCESS_KEY'.")
    print("Set it with e.g. `export KALSHI_ACCESS_KEY=your_key_here` and restart the notebook.")
    sys.exit(1)

try:
    private_key = load_private_key(PRIVATE_KEY_PATH)
except Exception as e:
    print(f"ERROR: Could not load private key from {PRIVATE_KEY_PATH}: {e}")
    sys.exit(1)

balance_path = "/trade-api/v2/portfolio/balance"
response = get_bal(private_key, API_KEY_ID, balance_path)
try:
    data = response.json()
except Exception as e:
    print("ERROR: Could not parse JSON response from Kalshi:", e)
    print("Raw response text:")
    print(response.text)
    sys.exit(1)

if "error" in data:
    print("Authentication or API error encountered:")
    print(data["error"].get("message", "Unknown error"))
    print(data)
else:
    print(data)


Authentication or API error encountered:
authentication_error
{'error': {'code': 'authentication_error', 'message': 'authentication_error', 'details': 'NOT_FOUND'}}
